# Bộ công cụ SEO tự động – chạy thật trên Google Colab

Notebook chạy cả 3 bài tập với **API thật** của Google:

| Bài | Nội dung | Nguồn dữ liệu |
|---|---|---|
| 1 | Phát hiện Content Decay | GA4 |
| 2 | Cảnh báo Keyword Cannibalization | Google Search Console |
| 3 | Liên kết GA4 × PageSpeed Insights | GA4 + PageSpeed Insights API |

### Chọn 1 trong 2 chế độ (ô **Cấu hình** bên dưới)

- **`API`** – bạn có quyền vào **GA4 property** và **Search Console** của một website (của bạn, hoặc được thầy/cô/công ty cấp).
  Code gọi đúng **GA4 Data API** và **Search Console API** như đề yêu cầu. Đây là chế độ chuẩn để nộp bài.
- **`PUBLIC`** – chưa có website. Bài 1 và bài 3 dùng **dữ liệu GA4 thật của Google Merchandise Store**
  mà Google công khai trên BigQuery (`bigquery-public-data.ga4_obfuscated_sample_ecommerce`, từ 01/11/2020 đến 31/01/2021).
  Bài 3 vẫn gọi **PageSpeed Insights API thật**.
  > Tài khoản *GA4 demo* của Google **không** gọi được bằng Data API (Google báo lỗi quyền), nên BigQuery export là cách hợp lệ duy nhất để có số liệu GA4 thật khi không có website.
  > Search Console không có bộ dữ liệu công khai, nên **bài 2 ở chế độ PUBLIC chỉ chạy được bằng dữ liệu mô phỏng**.

👉 Chạy lần lượt từng ô từ trên xuống (**Runtime → Run all** cũng được).

## 0. Cài đặt

In [ ]:
import os, sys
REPO_URL = "https://github.com/trannguyenminhhai265-cyber/seo-tools.git"
REPO_DIR = "/content/seo-tools"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q

!pip install -q -r {REPO_DIR}/requirements.txt google-cloud-bigquery db-dtypes

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)
print("Sẵn sàng. Kết quả sẽ lưu ở", OUT)

## 1. Cấu hình

**Chế độ `PUBLIC`** – cần:
- `GCP_PROJECT_ID`: ID của một Google Cloud project (tạo miễn phí tại [console.cloud.google.com](https://console.cloud.google.com), dùng *BigQuery sandbox*, không cần thẻ).
- `PSI_API_KEY`: API key của PageSpeed Insights (Cloud Console → *APIs & Services* → *Enable* "PageSpeed Insights API" → *Credentials* → *Create credentials → API key*).

**Chế độ `API`** – cần thêm:
- File JSON của **service account** (sẽ được hỏi tải lên ở bước 2). Email của service account phải được thêm vào GA4 (*Viewer*) và GSC (*Restricted*).
- `GA4_PROPERTY_ID`, `GSC_SITE_URL`, `SITE_URL`.

🔑 Nên lưu key trong **Secrets** của Colab (biểu tượng chìa khoá ở thanh bên trái) với tên `PSI_API_KEY`, bật *Notebook access*, để key không lộ khi nộp notebook.

In [ ]:
MODE = "PUBLIC"          #@param ["PUBLIC", "API"]
GCP_PROJECT_ID = ""      #@param {type:"string"}
GA4_PROPERTY_ID = ""     #@param {type:"string"}
GSC_SITE_URL = ""        #@param {type:"string"}
SITE_URL = ""            #@param {type:"string"}
PSI_SLEEP_SECONDS = 2    #@param {type:"number"}

# Lấy PSI key từ Colab Secrets, nếu chưa có thì hỏi trực tiếp (ẩn ký tự)
try:
    from google.colab import userdata
    PSI_API_KEY = userdata.get("PSI_API_KEY")
except Exception:
    PSI_API_KEY = None
if not PSI_API_KEY:
    from getpass import getpass
    PSI_API_KEY = getpass("Nhập PageSpeed Insights API key (Enter để bỏ qua – dễ bị lỗi 429): ").strip() or None

assert MODE in ("PUBLIC", "API")
if MODE == "PUBLIC":
    assert GCP_PROJECT_ID, "Chế độ PUBLIC cần GCP_PROJECT_ID"
else:
    assert GA4_PROPERTY_ID and GSC_SITE_URL and SITE_URL, "Chế độ API cần GA4_PROPERTY_ID, GSC_SITE_URL, SITE_URL"
print("MODE =", MODE, "| PSI key:", "có" if PSI_API_KEY else "KHÔNG")

## 2. Xác thực Google

In [ ]:
from google.colab import auth, files

CREDENTIALS = None
if MODE == "PUBLIC":
    # Đăng nhập tài khoản Google của bạn để chạy truy vấn BigQuery (dữ liệu công khai)
    auth.authenticate_user(project_id=GCP_PROJECT_ID)
    print("Đã đăng nhập Google cho BigQuery.")
else:
    print("Chọn file JSON của service account...")
    up = files.upload()
    CREDENTIALS = "/content/service-account.json"
    with open(CREDENTIALS, "wb") as f:
        f.write(next(iter(up.values())))
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS
    import json
    print("Service account:", json.load(open(CREDENTIALS))["client_email"])
    print("-> Email này phải có quyền Viewer trên GA4 và quyền trên Search Console.")

## 3. Bài 1 – Content Decay

- **API**: lấy `sessions` theo `pagePath` 6 tháng gần nhất từ GA4 Data API, chia 3 tháng đầu / 3 tháng cuối.
- **PUBLIC**: bộ dữ liệu công khai chỉ có **3 tháng** (01/11/2020 → 31/01/2021), nên chia đôi thành
  GĐ1 = 01/11 → 16/12/2020 và GĐ2 = 17/12/2020 → 31/01/2021 (mỗi giai đoạn 46 ngày). Công thức và bộ lọc giữ nguyên:
  **% Giảm > 30%** và **Traffic GĐ1 > 100 sessions**.

In [ ]:
import pandas as pd
from datetime import date
from IPython.display import display

decay_path = f"{OUT}/content_decay_report.csv"
if MODE == "API":
    !python bai1_content_decay.py --property-id {GA4_PROPERTY_ID} --credentials {CREDENTIALS} --output {decay_path}
else:
    import bai1_content_decay as b1
    import ga4_bigquery_public as pub
    print("Đang lấy dữ liệu GA4 thật (Google Merchandise Store) từ BigQuery...")
    raw = pub.fetch_daily_page_sessions(GCP_PROJECT_ID)
    print(f"{raw['pagePath'].nunique():,} trang | {raw['traffic'].sum():,} page-sessions")
    report = b1.compute_content_decay(raw, gd2_start=date(2020, 12, 17))
    report.to_csv(decay_path, index=False, encoding="utf-8-sig")
    print(f"Tìm thấy {len(report)} trang bị Content Decay (giảm > 30%, GĐ1 > 100 sessions)")

decay = pd.read_csv(decay_path)
display(decay.head(30))

## 4. Bài 2 – Keyword Cannibalization

- **API**: lấy `query, page, clicks, impressions, position` 30 ngày gần nhất từ Search Console API.
- **PUBLIC**: Search Console không có dữ liệu công khai, nên ô này chạy **dữ liệu mô phỏng** để minh hoạ thuật toán.
  Khi có quyền Search Console thì chuyển sang chế độ `API`.

In [ ]:
cannibal_prefix = f"{OUT}/keyword_cannibalization_report"
if MODE == "API":
    !python bai2_keyword_cannibalization.py --site-url "{GSC_SITE_URL}" --credentials {CREDENTIALS} --output-prefix {cannibal_prefix} --top 15
else:
    print("⚠ CHƯA CÓ QUYỀN SEARCH CONSOLE -> CHẠY BẰNG DỮ LIỆU MÔ PHỎNG\n")
    !python bai2_keyword_cannibalization.py --demo --output-prefix {cannibal_prefix} --top 15

## 5. Bài 3 – GA4 × PageSpeed Insights

Top 50 landing page (30 ngày) → gọi **PageSpeed Insights API thật** (mobile) cho từng URL, có `time.sleep` giữa các lượt
và tự thử lại khi gặp lỗi 429 → merge theo URL → lọc **PageSpeed < 50 & Bounce Rate > 70%**.

⏱ Mỗi URL mất khoảng 10–30 giây, nên 50 URL cần khoảng **15–25 phút**. Kết quả được cache: nếu Colab bị ngắt, chạy lại ô này sẽ đo tiếp từ chỗ dừng.

> **Lưu ý ở chế độ PUBLIC:** các URL cũ của `shop.googlemerchandisestore.com` hiện **redirect** sang `shop.merch.google`.
> PageSpeed sẽ đo trang đích sau redirect, xem cột `Final_URL`. Hãy ghi rõ điều này trong báo cáo nộp bài.

In [ ]:
psi_cache = f"{OUT}/psi_cache.json"
if MODE == "API":
    if PSI_API_KEY:
        os.environ["PSI_API_KEY"] = PSI_API_KEY   # script tự đọc biến môi trường, không lộ key trong lệnh
    !python bai3_pagespeed_ga4.py --property-id {GA4_PROPERTY_ID} --site-url {SITE_URL} --credentials {CREDENTIALS} --sleep {PSI_SLEEP_SECONDS} --cache {psi_cache} --output-dir {OUT}
else:
    import requests
    import bai3_pagespeed_ga4 as b3
    import ga4_bigquery_public as pub
    print("BƯỚC 1: Top landing pages tháng 01/2021 từ BigQuery (GA4 thật)...")
    ga4 = b3.prepare_ga4(pub.fetch_top_landing_pages(GCP_PROJECT_ID), pub.PUBLIC_SITE_URL, top_n=50)
    display(ga4.head(10))

    print(f"\nBƯỚC 2: Gọi PageSpeed Insights cho {len(ga4)} URL (nghỉ {PSI_SLEEP_SECONDS}s giữa các lượt)...")
    session = requests.Session()
    psi = b3.run_psi_loop(ga4["URL"].tolist(),
                          lambda u: b3.fetch_psi(u, PSI_API_KEY, "mobile", session),
                          sleep_s=PSI_SLEEP_SECONDS, cache_path=psi_cache)

    print("\nBƯỚC 3: Merge theo URL")
    merged = b3.merge_data(ga4, psi)
    ok = merged["PageSpeed_Mobile"].notna()
    print(f"  {ok.sum()}/{len(merged)} URL có điểm; {b3.count_redirects(merged)} URL bị redirect")
    if ok.sum() >= 3:
        print("  Pearson r (PageSpeed vs Bounce):",
              round(merged.loc[ok, "PageSpeed_Mobile"].corr(merged.loc[ok, "Bounce_Rate_%"]), 3))

    alert = b3.red_alert(merged)
    merged.to_csv(f"{OUT}/ga4_pagespeed_merged.csv", index=False, encoding="utf-8-sig")
    alert.to_csv(f"{OUT}/pagespeed_red_alert.csv", index=False, encoding="utf-8-sig")
    if ok.sum() > 0:
        b3.plot_relationship(merged, f"{OUT}/pagespeed_vs_bounce.png")
    print(f"\nBƯỚC 4: 🚨 {len(alert)} TRANG BÁO ĐỘNG ĐỎ")
    display(alert)

In [ ]:
from IPython.display import Image
chart = f"{OUT}/pagespeed_vs_bounce.png"
if os.path.exists(chart):
    display(Image(chart))

## 6. Tải toàn bộ kết quả về máy

In [ ]:
import shutil
# Đóng gói thư mục outputs (file service account nằm ngoài nên không bị đưa vào)
zip_path = shutil.make_archive("/content/ket_qua_seo", "zip", OUT)
!ls -la {OUT}
files.download(zip_path)